# Análisis bivariado

In [1]:
from eda_utils import *

## Precio y área

In [2]:
m = viv[viv.estrato.isin(ESTRATOS)].sample(6000, random_state=1)
fig = px.scatter(m, x="area", y="precio_M", color="estrato_txt", log_x=True, log_y=True, opacity=0.6,
                 category_orders={"estrato_txt": NOMBRES_ESTRATO}, color_discrete_sequence=COLORES_ESTRATO,
                 custom_data=["barrio_txt", "precio_txt"],
                 labels={"area": "área (m², escala log)", "precio_M": "precio (millones de 2025, escala log)",
                         "estrato_txt": ""})
fig.update_traces(marker_size=5, hovertemplate="<b>%{customdata[0]}</b><br>%{x:.0f} m²<br>%{customdata[1]}<extra></extra>")
mostrar(fig, "Precio y área", alto=520)

rho, p = stats.spearmanr(viv.precio_real, viv.area)
z, ee = np.arctanh(rho), 1 / np.sqrt(len(viv) - 3)
li, ls = np.tanh([z - 1.96 * ee, z + 1.96 * ee])
anotar("¿El precio sube con el área?", "Correlación de Spearman", rho, p,
       f"rho = {rho:.2f} (IC 95%: {li:.2f} a {ls:.2f})")

Correlación de Spearman: estadístico = 0.577   p = < 0,001   rho = 0.58 (IC 95%: 0.57 a 0.58)


A más área, más precio (Spearman 0,58), pero para un mismo tamaño el precio cambia
mucho según el estrato: los puntos más oscuros (estrato 6) quedan arriba de los más claros en
todo el rango.

## Estrato

In [3]:
d = viv[viv.estrato.isin(ESTRATOS)]
fig = px.box(d, x="estrato_txt", y="pm2_M", color="estrato_txt", points=False,
             category_orders={"estrato_txt": NOMBRES_ESTRATO}, color_discrete_sequence=COLORES_ESTRATO,
             labels={"estrato_txt": "", "pm2_M": "millones por m² (pesos de 2025)"})
fig.update_yaxes(range=[0, d.pm2_M.quantile(0.99)])
fig.update_layout(showlegend=False)
mostrar(fig, "Precio por m² según estrato")

(d.groupby("estrato_txt")
   .agg(ventas=("precio", "size"), precio_mediano=("precio_real", "median"),
        m2_mediano=("precio_m2_real", "median"), area_mediana=("area", "median"), edad_mediana=("edad", "median"))
   .assign(precio_mediano=lambda t: t.precio_mediano.map(cop), m2_mediano=lambda t: t.m2_mediano.map(cop)))

,ventas,precio_mediano,m2_mediano,area_mediana,edad_mediana
estrato_txt,,,,,
Estrato 1,8187,$122.5 M,$2.4 M,48.00,26.00
Estrato 2,3255,$115.7 M,$1.5 M,65.00,29.00
Estrato 3,7837,$176.5 M,$3.0 M,61.00,8.00
Estrato 4,9677,$313.9 M,$4.5 M,74.00,6.00
Estrato 5,2354,$342.2 M,$3.6 M,95.50,11.00
Estrato 6,2169,$672.5 M,$4.5 M,141.00,8.00


In [4]:
H, p, eps2 = kruskal_por_grupo(viv, "precio_m2_real", "estrato", ESTRATOS)
anotar("¿El precio por m² difiere entre estratos?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")

x4 = viv.loc[viv.estrato == "Medio_4", "precio_m2_real"]
x5 = viv.loc[viv.estrato == "Medio_Alto_5", "precio_m2_real"]
U, p = stats.mannwhitneyu(x4, x5)
anotar("¿El estrato 5 es más caro por m² que el 4?", "Mann-Whitney", U, p,
       f"medianas {cop(x4.median())} y {cop(x5.median())}")

Kruskal-Wallis: estadístico = 10,164.115   p = < 0,001   épsilon² = 0.30
Mann-Whitney: estadístico = 13,712,817.500   p = < 0,001   medianas $4.5 M y $3.6 M


El estrato explica el 30% de la variación del precio por m² (Kruskal-Wallis,
épsilon² = 0,30). Hay dos excepciones al orden esperado: el estrato 1 es más caro por m² que el
2 (2,4 M frente a 1,5 M), y el estrato 4 más caro que el 5 (4,5 M frente a 3,6 M; Mann-Whitney,
p < 0,001). Ninguna de las dos dice que vivir en un estrato más bajo cueste más: las dos salen de
**qué** se vende en cada estrato. Las secciones siguientes lo muestran.

### **¿Por qué el estrato 4 cuesta más por m² que el 5?**

Comparamos los dos estratos con viviendas parecidas: apartamentos de la misma antigüedad, del
mismo tamaño y de la misma fuente.

In [5]:
# los tonos del libro: claro para el estrato más bajo y oscuro para el más alto, como en el resto
# de los gráficos de estrato; la diferencia de luminosidad los separa también con daltonismo
COLOR_PAR = ["#7890A3", OSCURO]

def comparar_estratos(datos, estratos, paneles, titulo):
    """Precio por m² mediano de dos estratos, en grupos comparables, en paneles lado a lado."""
    fig = make_subplots(1, len(paneles), subplot_titles=[t for t, _, _ in paneles],
                        shared_yaxes=True, horizontal_spacing=0.05)
    for i, (_, columna, sub) in enumerate(paneles, 1):
        t = (sub.groupby([columna, "estrato_txt"], observed=True).pm2_M
                .agg(["median", "size"]).reset_index())
        for e, color in zip(estratos, COLOR_PAR):
            s = t[t.estrato_txt == e]
            fig.add_bar(x=s[columna].astype(str), y=s["median"], name=e, marker_color=color,
                        legendgroup=e, showlegend=i == 1, customdata=s["size"],
                        text=s["median"].round(1), textposition="outside", cliponaxis=False,
                        hovertemplate=f"<b>{e}</b><br>%{{x}}: %{{y:.2f}} M por m²"
                                      "<br>%{customdata:,} ventas<extra></extra>",
                        row=1, col=i)
    fig.update_layout(barmode="group", bargap=0.3, bargroupgap=0.08,
                      legend=dict(orientation="h", y=1.13, x=0), margin=dict(t=120))
    fig.update_yaxes(title_text="millones por m², pesos de 2025 (mediana)", row=1, col=1)
    mostrar(fig, titulo, alto=480)

par = viv[viv.estrato.isin(["Medio_4", "Medio_Alto_5"])].copy()
par["tipo"] = np.where(par.condicion_predio.eq("NPH"), "casa", "apartamento")
apt = par[par.tipo == "apartamento"].copy()
apt["antigüedad"] = pd.cut(apt.edad, [-1, 3, 10, 25, 200],
                           labels=["0-3 años", "4-10 años", "11-25 años", "más de 25"])
apt["tamaño"] = pd.cut(apt.area, [0, 60, 90, 130, np.inf],
                       labels=["< 60 m²", "60-90 m²", "90-130 m²", "> 130 m²"])
comparar_estratos(par, ["Estrato 4", "Estrato 5"],
                  [("Apartamentos por antigüedad", "antigüedad", apt),
                   ("Apartamentos por tamaño", "tamaño", apt),
                   ("Todas las viviendas por fuente", "fuente",
                    par[par.fuente.isin(["observatorio", "georreferenciadas", "igac"])])],
                  "Estrato 4 frente a estrato 5: precio por m² en viviendas comparables")

# qué vende cada estrato
a_2022 = ipc_anio.loc[2022] / ipc_anio
razon = par.precio * par.anio.map(a_2022) / par.avaluo
igac_con_avaluo = par.fuente.eq("igac") & par.avaluo.notna()
(par.assign(apto=par.tipo.eq("apartamento"),
            obra_nueva=par.tipo.eq("apartamento") & par.edad.le(3),
            igac=par.fuente.eq("igac"),
            al_avaluo=razon.between(0.9, 1.1).where(igac_con_avaluo))
    .groupby("estrato_txt")
    .agg(ventas=("pm2_M", "size"), **{"% apartamentos": ("apto", "mean"),
         "% obra nueva (0-3 años)": ("obra_nueva", "mean"),
         "área mediana apartamentos (m²)": ("area", lambda s: s[par.loc[s.index, "tipo"] == "apartamento"].median()),
         "antigüedad mediana apartamentos": ("edad", lambda s: s[par.loc[s.index, "tipo"] == "apartamento"].median()),
         "% ventas del IGAC": ("igac", "mean"),
         "% IGAC declarado al avalúo": ("al_avaluo", "mean")})
    .pipe(lambda t: t.assign(**{c: (t[c] * 100).round(0) for c in t.columns if c.startswith("%")}))
    .T)

estrato_txt,Estrato 4,Estrato 5
ventas,9677,2354
% apartamentos,93.00,94.00
% obra nueva (0-3 años),29.00,10.00
área mediana apartamentos (m²),72.00,92.00
antigüedad mediana apartamentos,6.00,11.00
% ventas del IGAC,60.00,68.00
% IGAC declarado al avalúo,30.00,40.00


 Esto pasa porque en el estrato 4 se venden muchas obras nuevas. El 29 % de sus ventas son apartamentos de 0 a 3 años, frente al 10 % en el estrato 5, y la obra nueva suele venderse más cara por m². Cuando se comparan apartamentos de la misma antigüedad la diferencia se reduce, e incluso en los de más de 10 años el estrato 5 resulta más caro (3,5 M frente a 3,3 M entre 11 y 25 años, y 2,8 M frente a 2,5 M en los de más de 25).


También influye el tamaño. Los apartamentos del estrato 5 son más grandes (92 m² frente a 72 m² de mediana), y entre más grande es un apartamento, más bajo suele ser su precio por m². 


### ¿Y por qué el estrato 1 cuesta más por m² que el 2?

In [6]:
par12 = viv[viv.estrato.isin(["Bajo_Bajo_1", "Bajo_2"])].copy()
par12["tipo"] = np.where(par12.condicion_predio.eq("NPH"), "casa", "apartamento")
comp = (par12.groupby("estrato_txt").tipo.value_counts(normalize=True).mul(100)
             .rename("pct").reset_index())
precio = (par12.groupby(["estrato_txt", "tipo"]).pm2_M.agg(["median", "size"]).reset_index())

fig = make_subplots(1, 2, subplot_titles=("Qué se vende: % de las ventas",
                                          "Precio por m² según el tipo de vivienda"),
                    horizontal_spacing=0.12)
for e, color in zip(["Estrato 1", "Estrato 2"], COLOR_PAR):
    s = comp[comp.estrato_txt == e]
    fig.add_bar(x=s.tipo, y=s.pct, name=e, marker_color=color, legendgroup=e,
                text=s.pct.round(0).astype(int).astype(str) + "%", textposition="outside",
                cliponaxis=False, hovertemplate=f"<b>{e}</b><br>%{{x}}: %{{y:.0f}}% de las ventas<extra></extra>",
                row=1, col=1)
    s = precio[precio.estrato_txt == e]
    fig.add_bar(x=s.tipo, y=s["median"], name=e, marker_color=color, legendgroup=e, showlegend=False,
                customdata=s["size"], text=s["median"].round(2), textposition="outside", cliponaxis=False,
                hovertemplate=f"<b>{e}</b><br>%{{x}}: %{{y:.2f}} M por m²<br>%{{customdata:,}} ventas<extra></extra>",
                row=1, col=2)
fig.update_layout(barmode="group", bargap=0.35, bargroupgap=0.08,
                  legend=dict(orientation="h", y=1.13, x=0), margin=dict(t=120))
fig.update_yaxes(title_text="% de las ventas del estrato", row=1, col=1)
fig.update_yaxes(title_text="millones por m², pesos de 2025 (mediana)", row=1, col=2)
mostrar(fig, "Estrato 1 frente a estrato 2: lo que cambia es qué se vende, no el precio", alto=460)

print("Apartamentos del estrato 1 por barrio:",
      par12[(par12.estrato_txt == "Estrato 1") & (par12.tipo == "apartamento")].barrio.value_counts().head(4).to_dict())

Apartamentos del estrato 1 por barrio: {'RURAL': 4196, 'VILLAS DE SAN PABLO': 1230, 'SECTOR ALAMEDA DEL RIO': 1162, 'LAS CAYENAS': 168}


Un apartamento cuesta 2,44 M por m² en el estrato 1 y 2,43 M en el 2, y una casa cuesta 1,13 M y 1,12 M, así que por tipo de vivienda los precios son prácticamente iguales. La diferencia aparece por lo que se vende en cada estrato. En el estrato 1 el 90 % de las ventas son apartamentos, mientras que en el estrato 2 solo el 45 %, porque allí predominan las casas. Como una casa vale por m² menos de la mitad que un apartamento (el lote y el área construida más grande bajan el precio por m²), el estrato 2 termina con un precio promedio más bajo.


## Barrio y localidad

Usamos solo las ventas con el área medida en la unidad, porque el área tomada del
edificio distorsiona el precio por m².

In [7]:
medidas = viv[viv.origen_caract == "unidad"]   # área medida en la unidad, no tomada del edificio
por_barrio = (medidas[medidas.barrio.notna() & ~medidas.barrio.isin(NO_BARRIOS)]
              .groupby("barrio").agg(n=("precio", "size"), pm2=("precio_m2_real", "median"),
                                     estrato=("estrato_num", "median"))
              .query("n >= 40").sort_values("pm2"))

fig = make_subplots(1, 2, subplot_titles=("15 más baratos", "15 más caros"), horizontal_spacing=0.28)
for i, t in enumerate([por_barrio.head(15), por_barrio.tail(15)], 1):
    fig.add_bar(x=t.pm2 / 1e6, y=t.index.str.title(), orientation="h", row=1, col=i,
                marker_color=PRINCIPAL if i == 1 else OSCURO, customdata=np.stack([t.n, t.estrato], axis=1),
                hovertemplate="<b>%{y}</b><br>%{x:.2f} M por m²<br>%{customdata[0]} ventas<br>"
                              "estrato mediano %{customdata[1]}<extra></extra>")
fig.update_xaxes(title_text="millones por m² (pesos de 2025)")
fig.update_layout(showlegend=False)
mostrar(fig, f"Precio mediano por m² en los barrios con 40 ventas o más ({len(por_barrio)} barrios)", alto=520)

print(f"Diferencia entre el más caro y el más barato: {por_barrio.pm2.max() / por_barrio.pm2.min():.1f} veces")

H, p, eps2 = kruskal_por_grupo(medidas, "precio_m2_real", "barrio", por_barrio.index)
anotar("¿El precio por m² difiere entre barrios?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")
localidades = [l for l in viv.localidad.dropna().unique() if l != "RURAL"]
H, p, eps2 = kruskal_por_grupo(viv, "precio_m2_real", "localidad", localidades)
anotar("¿El precio por m² difiere entre localidades?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")

Diferencia entre el más caro y el más barato: 7.7 veces
Kruskal-Wallis: estadístico = 13,275.010   p = < 0,001   épsilon² = 0.54
Kruskal-Wallis: estadístico = 9,639.479   p = < 0,001   épsilon² = 0.39


En los barrios más baratos el m² cuesta entre 0,8 y 1,2 M, mientras que en los más caros va de 4 a 6,3 M. Los primeros lugares los ocupan barrios con construcciones relativamente nuevas, como la Zona de Expansión Riomar, Palmas del Río y Paseo de la Castellana, donde la mayoría de los edificios tienen menos de 10 años.

## Evolución en el tiempo

In [8]:
serie = viv.groupby("anio").agg(n=("precio", "size"), nominal=("precio_m2", "median"),
                                 real=("precio_m2_real", "median"))
lineas = serie[serie.n >= 100]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=serie.index, y=serie.n, name="ventas", marker_color="#E3E8EC", secondary_y=False,
            hovertemplate="%{x}: %{y} ventas<extra></extra>")
fig.add_scatter(x=lineas.index, y=lineas.nominal / 1e6, name="en pesos de cada año", mode="lines+markers",
                line=dict(color=CLARO, width=2, dash="dot"), hovertemplate="%{x}: %{y:.2f} M<extra></extra>",
                secondary_y=True)
fig.add_scatter(x=lineas.index, y=lineas.real / 1e6, name="en pesos de 2025", mode="lines+markers",
                line=dict(color=OSCURO, width=3), hovertemplate="%{x}: %{y:.2f} M<extra></extra>",
                secondary_y=True)
fig.update_yaxes(title_text="ventas", secondary_y=False, showgrid=False, side="right")
fig.update_yaxes(title_text="millones por m² (mediana)", secondary_y=True, side="left",
                 tickmode="auto", tickformat=".1f", showgrid=True)
fig.update_xaxes(dtick=1)
fig.update_layout(legend=dict(orientation="h", y=1.08))
mostrar(fig, "Precio mediano por m² por año, con y sin inflación")

recientes = viv[viv.anio >= 2019]
H, p, eps2 = kruskal_por_grupo(recientes, "precio_m2_real", "anio", range(2019, 2026))
anotar("¿El m² real cambia entre 2019 y 2025?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.3f}")

serie.assign(nominal=serie.nominal.map(cop), real=serie.real.map(cop))

Kruskal-Wallis: estadístico = 490.465   p = < 0,001   épsilon² = 0.014


,n,nominal,real
anio,,,
2015,1,$2.7 M,$4.8 M
2016,56,$2.2 M,$3.8 M
2017,265,$3.0 M,$4.8 M
2018,408,$3.0 M,$4.6 M
2019,2366,$2.2 M,$3.2 M
2020,1202,$2.2 M,$3.2 M
2021,7399,$2.1 M,$2.9 M
2022,10237,$2.2 M,$2.8 M
2023,8459,$2.5 M,$2.9 M


En pesos corrientes el m² parecía subir 48% entre 2022 y 2025. En pesos de 2025 la
mediana baja de 3,2 M en 2019 a 2,8 M en 2022 y vuelve a 3,3 M en 2025. Parte de esos
movimientos no es cambio de precio sino de qué se vendió cada año (más o menos obra nueva,
más o menos estrato 6).

## Características de la vivienda

In [9]:
fig = make_subplots(1, 2, subplot_titles=("Precio según número de baños", "Precio por m² según piso"))
for b in range(1, 6):
    fig.add_box(y=viv.loc[viv.banios == b, "precio_M"], name=str(b), boxpoints=False,
                marker_color=PRINCIPAL, row=1, col=1)
g = viv[viv.piso.between(1, 15)].groupby(viv.piso.round()).pm2_M.median()
fig.add_scatter(x=g.index, y=g.values, mode="lines+markers", line=dict(color=OSCURO, width=3), row=1, col=2,
                hovertemplate="piso %{x}: %{y:.2f} M por m²<extra></extra>")
fig.update_yaxes(title_text="millones de 2025", range=[0, viv.precio_M.quantile(0.97)], row=1, col=1)
fig.update_yaxes(title_text="millones por m²", row=1, col=2)
fig.update_xaxes(title_text="baños", row=1, col=1)
fig.update_xaxes(title_text="piso", row=1, col=2)
fig.update_layout(showlegend=False)
mostrar(fig, "Baños y piso frente al precio")

filas = []
for col in ["banios", "habitaciones", "plantas", "piso", "edad"]:
    d = viv[[col, "precio_real", "precio_m2_real"]].dropna()
    r1, p1 = stats.spearmanr(d[col], d.precio_real)
    r2, p2 = stats.spearmanr(d[col], d.precio_m2_real)
    filas.append({"variable": col, "rho con precio": round(r1, 2), "p": p_texto(p1),
                  "rho con precio/m²": round(r2, 2), "p ": p_texto(p2)})
pd.DataFrame(filas).set_index("variable")

,rho con precio,p,rho con precio/m²,p
variable,,,,
banios,0.55,"< 0,001",0.34,"< 0,001"
habitaciones,0.36,"< 0,001",0.03,"< 0,001"
plantas,0.03,"< 0,001",-0.21,"< 0,001"
piso,0.34,"< 0,001",0.55,"< 0,001"
edad,-0.31,"< 0,001",-0.44,"< 0,001"


Los baños son la característica que más se relaciona con el precio (0,55). El piso se
relaciona más con el precio por m² (0,55) que con el precio total (0,34): los pisos altos
son apartamentos de zonas caras. Las viviendas más viejas son más baratas por m²
(−0,44). El número de plantas casi no importa para el precio (0,03).

El piso está vacío en el 12% de las viviendas: las que no tienen unidad propia en el catastro
y cuya dirección no dice el número de apartamento. Antes se rellenaba con la mediana del
edificio, pero el piso es de cada apartamento y esa mediana acertaba menos del 5%.

## Correlaciones

In [10]:
numericas = {"log_precio": "log precio", "area": "área", "precio_m2_real": "precio/m²", "avaluo": "avalúo",
             "estrato_num": "estrato", "banios": "baños", "habitaciones": "habitaciones", "piso": "piso",
             "edad": "edad", "lat": "latitud", "lon": "longitud", "anio": "año"}
corr = viv[list(numericas)].rename(columns=numericas).corr(method="spearman").round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale=ESCALA_DIVERGENTE, zmin=-1, zmax=1, aspect="auto")
fig.update_traces(textfont_size=11)
mostrar(fig, "Correlación de Spearman entre variables numéricas", alto=620)

Con el precio se relacionan sobre todo el precio por m² (0,73), el estrato (0,72), la
latitud (0,63) y el avalúo (0,62). Luego vienen el área (0,58) y los baños (0,55). Estrato,
latitud y avalúo dicen casi lo mismo sobre la ubicación, lo que genera colinealidad en un modelo
lineal.